Thin egraph + 

Thinning Terms
A notion of terms with rigid variables


Thinning Unification
Metmath0 bound variables look a lot like thinnings. There is a bitvector of what is allow to appear or not


Nominal Unification
Alpha prolog


Binders. Do they work?

Mcbride unification paper. Thinning are evidence of termination of telescoped substitution?
Unification over thinterms. A different sense than I was intending. I was considering thinvars are rigid,  forward existentials



# Thin

In [11]:
from dataclasses import dataclass, field
type Thin = list[bool]
def dom(f : Thin): # domain is big side
    return len(f)
def cod(f : Thin): # codomain is small side
    return sum(f)
def comp(f : Thin, g : Thin) -> Thin:
    assert cod(f) == dom(g)
    i = 0 
    result = []
    for a in f:
        if a:
            result.append(g[i])
            i += 1
        else:
            result.append(False)
    assert i == len(g)
    return result   
def lcm(f : Thin, g : Thin) -> Thin:
    assert len(f) == len(g)
    return [a and b for a,b in zip(f,g)]
def div(f : Thin, g : Thin) -> Thin:
    assert dom(f) == dom(g)
    assert all(not a for a,b in zip(f,g) if not b) # f is thinner than g
    return [a for a,b in zip(f,g) if b]

type Id = tuple[Thin, int]
@dataclass
class ThinUF:
    parents : list[Id] = field(default_factory=list)
    def makeset(self, scope : int) -> Id:
        i = len(self.parents)
        id = ([True]*scope, i)
        self.parents.append(id)
        return id
    def find(self, x : Id) -> Id:
        thin, xid = x
        while True:
            (thiny, yid) = self.parents[xid]
            if xid == yid:
                assert all(thiny)
                return (thin, xid)
            thin = comp(thiny, thin)
            xid = yid
    def union(self, x : Id, y : Id) -> Id | None:
        thinx, xid = self.find(x)
        thiny, yid = self.find(y)
        if xid != yid or thinx == thiny: 
            self.parents[xid] = yid
            return (thinx, xid)
        elif xid != yid or thinx != thiny:
            thinz = lcm(thinx,thiny)
            (_, z) = self.makeset(cod(thinz))
            self.parents[xid] = (div(thinz,thinx), z)
            self.parents[yid] = (div(thinz,thiny), z)
            return (thinz, z)
        else:
            # hmm
            return None
type Thin = tuple[bool, ...]
type Id = tuple[Thin, int]
def ctx(id : Id):
    return dom(id[0])
def widen(thin, x : Id) -> Id: # weaken
    return (comp(thin,x[0]), x[1])
# Based on https://github.com/mwillsey/microegg
class Pattern: ...
@dataclass(frozen=True)
class Var(Pattern):
    name: str
@dataclass(frozen=True)
class PApp(Pattern):
    f: str
    args: tuple[Pattern, ...]
type Node = tuple[str, tuple[Id, ...]]
type Subst = dict[str, Id]

# egraph

In [ ]:
@dataclass(frozen=True)
class Node:
    f : str
    args : tuple[Id,...]

@dataclass
class EGraph:
    uf: ThinUF = field(default_factory=ThinUF)
    memo: dict[Node, Id] = field(default_factory=dict)


    def add_node(self, obj: Node, n=0) -> Id:
        id = self.memo.get(obj)
        if id is not None:
            return self.find(id)
        else:
            id = self.uf.makeset(n)
            self.memo[obj] = id
            return id

    def app(self, f: str, *args: Id) -> Id:
        assert all(ctx(arg) == ctx(args[0]) for arg in args)
        # todo lift pulling
        return self.add_node(Node(f, args))

    def find(self, id: Id) -> Id:
        return self.uf.find(id)
    def union(self, id1: Id, id2: Id):
        return self.uf.union(id1, id2)        
    def nodes_in_class(self, id: Id) -> list[object]:
        id = self.find(id)
        return [obj for obj, obj_id in self.memo.items() if self.find(obj_id) == id]
    def is_eq(self, a: Id, b: Id) -> bool:
        return self.find(a) == self.find(b)
    def canonize_node(self, node: Node) -> Node:
        return Node(node.f, tuple(self.find(arg) for arg in node.args))

    def rebuild(self):
        copy_memo = self.memo.copy()
        while True:
            done = True
            for obj, id in copy_memo.items():
                id = self.find(id)
                new_node = self.canonize_node(obj)
                new_id = self._add(new_node)
                if new_id != id:
                    self.union(id, new_id)
                    done = False
            if done:
                return

    def ematch(self, pattern: Pattern, id: Id) -> list[Subst]:
        return self.ematch_rec(pattern, id, {})

    def ematch_rec(self, pattern: Pattern, id: Id, subst: Subst) -> list[Subst]:
        id = self.find(id)
        match pattern:
            case Var(name):
                if name in subst:
                    if self.is_eq(subst[name], id):
                        return [subst]
                    else:
                        return []
                else:
                    return [{**subst, name: id}]
            case PApp(f, args):
                results = []
                for obj in self.nodes_in_class(id):
                    match obj:
                        case (f0, arg_ids) if f0 == f and len(arg_ids) == len(args):
                            todo = [subst]
                            for arg_pattern, arg_id in zip(args, arg_ids):
                                todo = [
                                    subst1
                                    for subst0 in todo
                                    for subst1 in self.ematch_rec(
                                        arg_pattern, arg_id, subst0
                                    )
                                ]
                            results.extend(todo)
                        case _:
                            raise ValueError(f"Unexpected object in e-graph: {obj}")
                return results


E = EGraph()
a,b,c = E.app("a"), E.app("b"), E.app("c")
E.union(a,b)
E.union(b,c)
E.is_eq(a,c)
E

TypeError: cannot unpack non-iterable int object